<a href="https://colab.research.google.com/github/riofutabac/PlacasVideos/blob/main/GoogleColab_ALPR_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Pipeline ALPR Orientado a Eventos — Vía de Lastre (Píntag)

**Arquitectura:** Crop físico + Motion Gate de 3 estados + YOLOv8 + ByteTrack + Top-M/Top-K + FastPlateOCR + Deduplicación + Reporte Excel con fotos incrustadas.

Repositorio: [`riofutabac/PlacasVideos`](https://github.com/riofutabac/PlacasVideos) (rama `main`).

> **Entorno de ejecución:** antes de correr cualquier celda, ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución` y selecciona **GPU (T4)**. Sin GPU, el pipeline funcionará pero mucho más lento.

## 1. Verificar GPU disponible

In [ ]:
# Comprobar que Colab asignó una GPU NVIDIA (T4) y que PyTorch la detecta
!nvidia-smi

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).")


## 2. Montar Google Drive

In [ ]:
# Los videos de la cámara viven en Google Drive, dentro de la carpeta "Cam PL"
from google.colab import drive

drive.mount("/content/drive")


## 3. Clonar o actualizar el repositorio

In [ ]:
import os

REPO_URL = "https://github.com/riofutabac/PlacasVideos.git"
REPO_DIR = "/content/PlacasVideos"
BRANCH = "main"

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("El repositorio ya existe, actualizando con git pull...")
    !cd "$REPO_DIR" && git fetch origin && git checkout "$BRANCH" && git pull origin "$BRANCH"

%cd $REPO_DIR
!git status


## 4. Instalar dependencias

Instalamos las dependencias del `requirements.txt` más las que Colab necesita de forma
específica: `onnxruntime-gpu` (compatible con CUDA 12, en vez del `onnxruntime` de CPU),
`fast-alpr`, y `onnx` (lo pide `ultralytics` para exportar, y si no está preinstalado
pide reiniciar el entorno de ejecución a mitad de la corrida).

In [ ]:
# Instalar dependencias base del proyecto
!pip install -q -r requirements.txt
!pip install -q fast-alpr onnx

# Evitar conflicto de CUDA 13 en Colab: usar onnxruntime-gpu compatible con CUDA 12
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.22.0

# Fijar versión de supervision compatible con el pipeline
!pip install -q "supervision<0.31" "opencv-python-headless<5"


In [ ]:
# Verificar versiones instaladas de las librerías clave
import cv2
import onnxruntime as ort
import supervision as sv
import ultralytics

print(f"OpenCV: {cv2.__version__}")
print(f"ONNX Runtime: {ort.__version__} | providers: {ort.get_available_providers()}")
print(f"Supervision: {sv.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")


## 5. Configuración

In [ ]:
# Carpeta de Drive donde están los videos de la cámara
VIDEOS_DIR = "/content/drive/MyDrive/Cam PL"

# Clips específicos para la corrida de validación rápida (60 y 61 tienen ground truth)
CLIPS = [60, 61]

# Límite de videos para una corrida de producción parcial (None = sin límite)
LIMIT = None


## 6. Corrida de validación rápida (clips 60 y 61)

Estos dos clips tienen datos de referencia (*ground truth*). Al incluirlos, `main.py`
imprime automáticamente al final un bloque **GROUND TRUTH EVALUATION** con precisión,
recall y las placas que coinciden o no contra el ground truth — úsalo para confirmar
que el pipeline sigue funcionando correctamente después de cualquier cambio.

In [ ]:
!python main.py "$VIDEOS_DIR" --clips {" ".join(map(str, CLIPS))}


## 7. Corrida de producción

Procesa toda la carpeta de videos, o usa `--limit` para procesar solo los primeros N
archivos (útil para pruebas intermedias antes de lanzar la carpeta completa).

In [ ]:
# Toda la carpeta:
!python main.py "$VIDEOS_DIR"

# O bien, limitar la cantidad de videos a procesar (descomenta y ajusta LIMIT en la celda de configuración):
# !python main.py "$VIDEOS_DIR" --limit 10


## 8. Descargar el reporte Excel de auditoría

In [ ]:
import os
from google.colab import files

report_path = "reports/reporte_auditoria.xlsx"
if os.path.exists(report_path):
    files.download(report_path)
    print(f"📥 Descargando {report_path}...")
else:
    print(f"⚠️ No se encontró {report_path}. Corre el pipeline primero (secciones 6 o 7).")


## 9. Benchmarks / Diagnóstico (opcional)

Las siguientes celdas **no son necesarias** para generar el reporte de auditoría.
Sirven para diagnosticar rendimiento y comparar backends de inferencia/decodificación.
Ejecuta solo la(s) que necesites.

### 9.1 Benchmark de runtime del detector de vehículos (requiere GPU)

In [ ]:
# Compara ultralytics vs ONNX Runtime (I/O binding) vs TensorRT para el modelo YOLO
!python benchmarks/benchmark_vehicle_runtime.py --model yolov8n.onnx


### 9.2 Diagnóstico A/B de decodificadores de video (NVDEC vs OpenCV)

In [ ]:
# Compara el decodificador NVDEC (GPU) contra OpenCV (CPU) sobre el pipeline completo
!python benchmarks/diagnose_ab_decoders.py --full-pipeline


### 9.3 Benchmark puro de FFmpeg + NVDEC

In [ ]:
!python benchmarks/benchmark_ffmpeg_nvdec.py


### 9.4 Resumen ejecutivo de la última corrida

In [ ]:
# Resumen con métricas clave y costo estimado de GPU en la nube
!python benchmarks/resumen_ejecutivo.py --cost-per-hour 0.35


## 10. Solución de problemas

- **"archivo dañado" / error al abrir un clip:** revisa el error real impreso *justo
  arriba* de ese mensaje en la salida de la celda — normalmente indica la causa
  concreta (códec no soportado, archivo incompleto en Drive, etc.), y el mensaje de
  "archivo dañado" es solo el resumen final.
- **Cambiaste dependencias (paso 4) y algo falla de forma rara:** reinicia el entorno
  de ejecución (`Entorno de ejecución → Reiniciar entorno de ejecución`) y vuelve a
  ejecutar desde el paso 2 en adelante. Reinstalar `onnxruntime-gpu` u `onnx` sin
  reiniciar puede dejar el proceso de Python con la versión vieja cargada en memoria.
- **`git pull` falla con conflictos:** el repo en `/content/PlacasVideos` puede tener
  cambios locales de una corrida anterior. Bórralo (`!rm -rf /content/PlacasVideos`) y
  vuelve a ejecutar el paso 3 para clonar limpio.
- **No aparece el bloque GROUND TRUTH EVALUATION:** solo se imprime cuando la corrida
  incluye los clips 60 y/o 61 (`--clips 60 61`), porque son los únicos con datos de
  referencia.